In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

In [0]:
df=spark.createDataFrame(data,columns)
display(df)


order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### Requirement 1: Sales Analysis
### Total sales by product
### Total sales by category
### Total sales by city

In [0]:
from pyspark.sql.functions import sum, col, when

df = df.withColumn("amount", col("amount").cast("int"))

df = df.withColumn(
    "amount",
    when(col("amount").isNull(), 0).otherwise(col("amount"))
)

df = df.filter(col("amount") >= 0)



In [0]:
product_sales = df.groupBy("product") \
    .agg(sum("amount").alias("Total_sales"))

display(product_sales)

product,Total_sales
Laptop,105000
Mobile,48000
Tablet,42000
Watch,8000
Headphones,3000


In [0]:
category_sales=df.groupBy("category") \
    .agg(sum("amount").alias("Total_sales"))

display(category_sales)

category,Total_sales
Electronics,195000
Accessories,11000


In [0]:
df=df.withColumn("city", when(col("city").isNull(), "Unknown").otherwise(col("city")))
city_sales=df.groupBy("city").agg(sum("amount").alias("Total_sales"))

display(city_sales)


city,Total_sales
Hyderabad,92000
Chennai,48000
Delhi,55000
Mumbai,8000
Unknown,3000


### Requirement 2: Customer Insights
### Number of orders per customer
### Top customers based on spending

In [0]:
from pyspark.sql.functions import count
No_of_orders=df.groupBy("customer_id").agg(count("order_id").alias("No_of_orders"))
display(No_of_orders)


customer_id,No_of_orders
C001,3
C002,2
C003,1
C004,1
C005,1
C007,2


In [0]:
top_customers=df.groupBy("customer_id").agg(sum("amount").alias("Total_amount")).orderBy(col("Total_amount").desc())
display(top_customers)


customer_id,Total_amount
C001,92000
C003,55000
C007,30000
C002,18000
C004,8000
C005,3000


### Requirement 3: Data Quality
### No NULL values in critical columns
### No duplicate orders
### No negative sales

In [0]:

from pyspark.sql.functions import col

df_clean = df.dropna(subset=["order_id", "customer_id", "product", "amount"]) \
             .dropDuplicates(["order_id"]) \
             .filter(col("amount") >= 0)
display(df_clean)

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### Requirement 4: Latest Data Only
### If same order_id appears multiple times → keep latest record

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())
df = df.withColumn("row_num", row_number().over(window_spec))
df_latest = df.filter(col("row_num") == 1).drop("row_num")
display(df_latest)

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


### Requirement 5: Dashboard Ready Data
### Data should be clean and aggregated
### Ready for tools like Power BI

In [0]:
from pyspark.sql.functions import sum

product_sales = df_clean.groupBy("product") \
    .agg(sum("amount").alias("total_sales"))

In [0]:
city_sales = df_clean.groupBy("city") \
    .agg(sum("amount").alias("total_sales"))

In [0]:
daily_sales = df_clean.groupBy("date") \
    .agg(sum("amount").alias("daily_sales"))

In [0]:
customer_sales = df_clean.groupBy("customer_id") \
    .agg(sum("amount").alias("total_spent"))

In [0]:
product_sales.write.format("delta").mode("overwrite").saveAsTable("gold_product_sales")

city_sales.write.format("delta").mode("overwrite").saveAsTable("gold_city_sales")

daily_sales.write.format("delta").mode("overwrite").saveAsTable("gold_daily_sales")

customer_sales.write.format("delta").mode("overwrite").saveAsTable("gold_customer_sales")